# 🚁 SAHI + Deformable DETR — AU-AIR Fine-Tuning


## Google Drive Connection

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
IMAGES_DIR = '/content/drive/MyDrive/DI725_assignment2_e274020/auair_dataset/images'
ANNOT_FILE = '/content/drive/MyDrive/DI725_assignment2_e274020/auair_dataset/annotations.json'
OUTPUT_DIR = '/content/drive/MyDrive/DI725_assignment2_e274020/output_sahi_3'

import os, shutil
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
%%capture
!pip install sahi transformers accelerate timm pycocotools -q

## Import Libraries

In [4]:
import os, json, random, time, tempfile, shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    get_cosine_schedule_with_warmup,
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


In [5]:

CATEGORIES   = ['Human', 'Car', 'Truck', 'Van', 'Motorbike', 'Bicycle', 'Bus', 'Trailer']
NUM_CLASSES  = len(CATEGORIES)   # 8
SEED         = 42
VAL_RATIO    = 0.1

# SAHI dilimleme parametreleri
SLICE_H      = 640     # Tile yüksekliği (drone görüntüsü için 640 ideal)
SLICE_W      = 640     # Tile genişliği
OVERLAP_H    = 0.2     # Dikey örtüşme oranı
OVERLAP_W    = 0.2     # Yatay örtüşme oranı
MIN_AREA     = 0.1     # Tile kenarındaki bbox minimum alan oranı

# Eğitim parametreleri
BATCH_SIZE   = 16
NUM_EPOCHS   = 10
LR_BACKBONE  = 1e-5
LR_HEAD      = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.08
IMG_SHORT    = 480     # Processor resize (slice zaten 640 olduğu için büyük değere gerek yok)
IMG_LONG     = 640
GRAD_CLIP    = 0.1
SAVE_EVERY   = 2
USE_AMP      = True

MODEL_NAME   = 'SenseTime/deformable-detr'
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Local çalışma dizinleri
COCO_DIR      = 'content/drive/MyDrive/DI725_assignment2_e274020/auair_coco'      # AU-AIR → COCO dönüşümü
SLICED_DIR    = '/content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2'    # SAHI tile çıktıları
#os.makedirs(COCO_DIR,   exist_ok=True)
#os.makedirs(SLICED_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'SLICE  : {SLICE_H}x{SLICE_W} | overlap {OVERLAP_H}')
print(f'AMP    : {USE_AMP} | Batch: {BATCH_SIZE} | Epochs: {NUM_EPOCHS}')

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB
SLICE  : 640x640 | overlap 0.2
AMP    : True | Batch: 16 | Epochs: 10


## 4️⃣ AU-AIR JSON → Standard COCO JSON

In [ ]:
def auair_to_coco(annot_path, images_dir, val_ratio=0.1, seed=42):
    """AU-AIR özel formatını COCO'ya çevirir ve train/val böler."""
    with open(annot_path) as f:
        raw = json.load(f)

    coco_cats = [
        {'id': i+1, 'name': c, 'supercategory': 'object'}
        for i, c in enumerate(raw['categories'])
    ]

    images_list, anns_list = [], []
    img_id = ann_id = 1

    for record in raw['annotations']:
        fname = record['image_name']
        fpath = os.path.join(images_dir, fname)
        if not os.path.exists(fpath):
            continue

        W = int(record.get('image_width:', record.get('image_width', 1920)))
        H = int(record['image_height'])
        images_list.append({'id': img_id, 'file_name': fname, 'width': W, 'height': H})

        for b in record.get('bbox', []):
            x1 = max(0, b['left'])
            y1 = max(0, b['top'])
            x2 = min(W, b['left'] + b['width'])
            y2 = min(H, b['top']  + b['height'])
            bw, bh = x2 - x1, y2 - y1
            if bw <= 0 or bh <= 0:
                continue
            anns_list.append({
                'id': ann_id, 'image_id': img_id,
                'category_id': int(b['class']) + 1,
                'bbox': [x1, y1, bw, bh],
                'area': bw * bh, 'iscrowd': 0
            })
            ann_id += 1
        img_id += 1

    # Train/val bölme
    random.seed(seed)
    all_ids = [i['id'] for i in images_list]
    random.shuffle(all_ids)
    val_n   = max(1, int(len(all_ids) * val_ratio))
    val_ids = set(all_ids[:val_n])
    trn_ids = set(all_ids[val_n:])

    def split(ids):
        return {
            'info': {'description': 'AU-AIR COCO'},
            'categories': coco_cats,
            'images':      [i for i in images_list if i['id'] in ids],
            'annotations': [a for a in anns_list   if a['image_id'] in ids],
        }

    tr, vl = split(trn_ids), split(val_ids)
    print(f'Train: {len(tr["images"]):,} images  {len(tr["annotations"]):,} annotation')
    print(f'Val  : {len(vl["images"]):,} images  {len(vl["annotations"]):,} annotation')
    return tr, vl


print('AU-AIR → COCO ...')
train_coco_raw, val_coco_raw = auair_to_coco(
    ANNOT_FILE, IMAGES_DIR, VAL_RATIO, SEED
)

# Fix COCO_DIR path and create directory
corrected_coco_dir = COCO_DIR if COCO_DIR.startswith('/') else '/' + COCO_DIR
os.makedirs(corrected_coco_dir, exist_ok=True)

# COCO JSON'larını diske kaydet (slice_coco() dosya yolu ister)
TRAIN_COCO_RAW = os.path.join(corrected_coco_dir, 'train_raw.json')
VAL_COCO_RAW   = os.path.join(corrected_coco_dir, 'val_raw.json')
json.dump(train_coco_raw, open(TRAIN_COCO_RAW, 'w'))
json.dump(val_coco_raw,   open(VAL_COCO_RAW,   'w'))
print('Raw COCO JSON is saved.')

AU-AIR → COCO ...
Train: 29,540 images  118,847 annotation
Val  : 3,282 images  13,127 annotation
Raw COCO JSON is saved.


## SAHI slicing

In [ ]:
from sahi.slicing import slice_coco

SLICED_TRAIN_DIR = os.path.join(SLICED_DIR, 'train')
SLICED_VAL_DIR   = os.path.join(SLICED_DIR, 'val')
os.makedirs(SLICED_TRAIN_DIR, exist_ok=True)
os.makedirs(SLICED_VAL_DIR,   exist_ok=True)

print(f' Train dataset splitting ({SLICE_H}x{SLICE_W}, overlap={OVERLAP_H})...')
t0 = time.time()

train_sliced_dict, train_sliced_path = slice_coco(
    coco_annotation_file_path = TRAIN_COCO_RAW,
    image_dir                 = IMAGES_DIR,
    output_coco_annotation_file_name = 'train_sliced',
    output_dir                = SLICED_TRAIN_DIR,
    slice_height              = SLICE_H,
    slice_width               = SLICE_W,
    overlap_height_ratio      = OVERLAP_H,
    overlap_width_ratio       = OVERLAP_W,
    min_area_ratio            = MIN_AREA,
    ignore_negative_samples   = False,  # Nesnesiz tile'lar da dahil (background öğrenme)
    verbose                   = True
)
print(f'Train: {len(train_sliced_dict["images"]):,} tile  '
      f'{len(train_sliced_dict["annotations"]):,} ann  '
      f'({(time.time()-t0)/60:.1f} dk)')

print(f'Validation dataset splitting')
t0 = time.time()

val_sliced_dict, val_sliced_path = slice_coco(
    coco_annotation_file_path = VAL_COCO_RAW,
    image_dir                 = IMAGES_DIR,
    output_coco_annotation_file_name = 'val_sliced',
    output_dir                = SLICED_VAL_DIR,
    slice_height              = SLICE_H,
    slice_width               = SLICE_W,
    overlap_height_ratio      = OVERLAP_H,
    overlap_width_ratio       = OVERLAP_W,
    min_area_ratio            = MIN_AREA,
    ignore_negative_samples   = False,
    verbose                   = True
)
print(f'Val  : {len(val_sliced_dict["images"]):,} tile  '
      f'{len(val_sliced_dict["annotations"]):,} ann  '
      f'({(time.time()-t0)/60:.1f} dk)')

TRAIN_SLICED_JSON = train_sliced_path
VAL_SLICED_JSON   = val_sliced_path
print(f'\nTrain JSON : {TRAIN_SLICED_JSON}')
print(f'Val JSON   : {VAL_SLICED_JSON}')

 Train dataset splitting (640x640, overlap=0.2)...


Görüntülenen çıkış son 5000 satıra kısaltıldı.
INFO:sahi:image.shape: (1920, 1080)
2026-04-24 16:55:06,273 - sahi - INFO - Num slices: 8 slice_height: 640 slice_width: 640 (slicing.py:399)
INFO:sahi:Num slices: 8 slice_height: 640 slice_width: 640
 99%|█████████▉| 29292/29541 [2:48:28<01:29,  2.78it/s]2026-04-24 16:55:06,340 - sahi - INFO - sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/train/frame_20190906150731_xx_0001755_29291_1024_0_1664_640.png (slicing.py:317)
INFO:sahi:sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/train/frame_20190906150731_xx_0001755_29291_1024_0_1664_640.png
2026-04-24 16:55:06,356 - sahi - INFO - sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/train/frame_20190906150731_xx_0001755_29291_1280_0_1920_640.png (slicing.py:317)
INFO:sahi:sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/train/frame_20190906150731_xx_0001755_2

Train: 236,328 tile  238,535 ann  (170.7 dk)
Validation dataset splitting


Görüntülenen çıkış son 5000 satıra kısaltıldı.
INFO:sahi:image.shape: (1920, 1080)
2026-04-24 17:15:15,633 - sahi - INFO - Num slices: 8 slice_height: 640 slice_width: 640 (slicing.py:399)
INFO:sahi:Num slices: 8 slice_height: 640 slice_width: 640
 92%|█████████▏| 3033/3282 [18:05<01:28,  2.82it/s]2026-04-24 17:15:15,730 - sahi - INFO - sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/val/frame_20190906150731_x_0000628_3032_0_0_640_640.png (slicing.py:317)
INFO:sahi:sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/val/frame_20190906150731_x_0000628_3032_0_0_640_640.png
2026-04-24 17:15:15,740 - sahi - INFO - sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/val/frame_20190906150731_x_0000628_3032_0_440_640_1080.png (slicing.py:317)
INFO:sahi:sliced image path: /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/val/frame_20190906150731_x_0000628_3032_0_440_640_1080.png
2026-

Val  : 26,256 tile  26,131 ann  (19.7 dk)

Train JSON : /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/train/train_sliced_coco.json
Val JSON   : /content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/val/val_sliced_coco.json


## Statistics

In [ ]:
def dataset_stats(coco_dict, name):
    cats   = {c['id']: c['name'] for c in coco_dict['categories']}
    counts = defaultdict(int)
    areas  = []
    for ann in coco_dict['annotations']:
        counts[cats[ann['category_id']]] += 1
        areas.append(ann['area'])

    print(f'\n──── {name} ────')
    print(f'Toplam tile       : {len(coco_dict["images"]):,}')
    print(f'Toplam annotation : {len(coco_dict["annotations"]):,}')
    if areas:
        print(f'Ortalama bbox alan: {np.mean(areas):.0f} px²')
        print(f'Medyan  bbox alan : {np.median(areas):.0f} px²')
    print('Sınıf dağılımı:')
    for cls, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        print(f'  {cls:<12}: {cnt:>6,}')

dataset_stats(train_sliced_dict, 'TRAIN (sliced)')
dataset_stats(val_sliced_dict,   'VAL   (sliced)')

sample_imgs = random.sample(train_sliced_dict['images'], min(6, len(train_sliced_dict['images'])))
ann_by_img  = defaultdict(list)
for ann in train_sliced_dict['annotations']:
    ann_by_img[ann['image_id']].append(ann)
cat_map = {c['id']: c['name'] for c in train_sliced_dict['categories']}
COLORS  = plt.cm.Set1(np.linspace(0, 1, NUM_CLASSES + 1))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_rec in zip(axes.flat, sample_imgs):
    tile_path = os.path.join(SLICED_TRAIN_DIR, img_rec['file_name'])
    if not os.path.exists(tile_path):
        ax.axis('off'); continue
    img = Image.open(tile_path)
    ax.imshow(img)
    for ann in ann_by_img[img_rec['id']]:
        x, y, w, h = ann['bbox']
        cid   = ann['category_id']
        color = COLORS[cid % len(COLORS)]
        ax.add_patch(mpatches.Rectangle(
            (x, y), w, h, linewidth=1.5,
            edgecolor=color, facecolor='none'
        ))
        ax.text(x, y-3, cat_map[cid], color='white', fontsize=7,
                bbox=dict(facecolor=color, alpha=0.7, pad=1))
    ax.set_title(img_rec['file_name'], fontsize=7)
    ax.axis('off')

plt.suptitle(f'SAHI Tile Samples ({SLICE_H}×{SLICE_W}, overlap={OVERLAP_H})', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sahi_tile_samples.png'), dpi=120)
plt.show()
print(' Tile samples saved.')

##Dataset

In [6]:
class SlicedCOCODataset(Dataset):
    """
    SAHI tarafından oluşturulan tile görüntüler ve COCO annotation'ları ile çalışır.
    """
    def __init__(self, coco_json_path, tiles_dir, processor, is_train=True):
        with open(coco_json_path) as f:
            coco_dict = json.load(f)

        self.tiles_dir  = tiles_dir
        self.processor  = processor
        self.is_train   = is_train
        self.images     = {img['id']: img for img in coco_dict['images']}
        self.img_ids    = list(self.images.keys())
        self.anns       = defaultdict(list)
        for ann in coco_dict['annotations']:
            self.anns[ann['image_id']].append(ann)

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id  = self.img_ids[idx]
        rec     = self.images[img_id]

        # Tile dosya yolu
        tile_path = os.path.join(self.tiles_dir, rec['file_name'])
        image     = Image.open(tile_path).convert('RGB')
        W, H      = image.size

        # Augmentation (sadece train)
        flip = self.is_train and random.random() < 0.5
        if flip:
            image = T.functional.hflip(image)
        if self.is_train:
            image = T.ColorJitter(
                brightness=0.25, contrast=0.25, saturation=0.15, hue=0.08
            )(image)

        boxes, labels = [], []
        for ann in self.anns[img_id]:
            x, y, bw, bh = ann['bbox']
            if flip:
                x = W - x - bw
            boxes.append([x, y, bw, bh])
            labels.append(ann['category_id'])

        target = {
            'image_id': img_id,
            'annotations': [
                {'bbox': b, 'category_id': l, 'area': b[2]*b[3], 'iscrowd': 0}
                for b, l in zip(boxes, labels)
            ]
        }

        enc = self.processor(images=image, annotations=target, return_tensors='pt')
        return (
            enc['pixel_values'].squeeze(0),
            enc['pixel_mask'].squeeze(0),
            enc['labels'][0]
        )


def collate_fn(batch):
    return (
        torch.stack([b[0] for b in batch]),
        torch.stack([b[1] for b in batch]),
        [b[2] for b in batch]
    )

print('Dataset class is ready.')

Dataset class is ready.


##Processor, Model & DataLoader

In [7]:
import os

SLICED_TRAIN_DIR = os.path.join(SLICED_DIR, 'train')
SLICED_VAL_DIR   = os.path.join(SLICED_DIR, 'val')
TRAIN_SLICED_JSON = os.path.join(SLICED_TRAIN_DIR, 'train_sliced_coco.json')
VAL_SLICED_JSON   = os.path.join(SLICED_VAL_DIR, 'val_sliced_coco.json')

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME,
    size={'shortest_edge': IMG_SHORT, 'longest_edge': IMG_LONG}
)

id2label = {0: 'N/A'}
id2label.update({i+1: c for i, c in enumerate(CATEGORIES)})
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForObjectDetection.from_pretrained(
    MODEL_NAME,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(DEVICE)

train_ds = SlicedCOCODataset(TRAIN_SLICED_JSON, SLICED_TRAIN_DIR, processor, is_train=True)
val_ds   = SlicedCOCODataset(VAL_SLICED_JSON,   SLICED_VAL_DIR,   processor, is_train=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True,
    persistent_workers=True, prefetch_factor=2,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
    persistent_workers=True, prefetch_factor=2,
    collate_fn=collate_fn
)

print(f'Train tiles  : {len(train_ds):,}  ({len(train_loader):,} batch)')
print(f'Val   tiles  : {len(val_ds):,}  ({len(val_loader):,} batch)')
print(f'Model params : {sum(p.numel() for p in model.parameters()):,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `DeformableDetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a

Loading weights:   0%|          | 0/545 [00:00<?, ?it/s]

DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                                         | Status     |                                                                                      
----------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------
model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.laye

OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/DI725_assignment2_e274020/auair_sliced_2/train/train_sliced_coco.json'

##Optimizer, Scheduler & Resume

In [ ]:
backbone_params, head_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    (backbone_params if 'backbone' in name else head_params).append(param)

optimizer = torch.optim.AdamW(
    [{'params': backbone_params, 'lr': LR_BACKBONE},
     {'params': head_params,     'lr': LR_HEAD}],
    weight_decay=WEIGHT_DECAY
)

total_steps  = NUM_EPOCHS * len(train_loader)
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = GradScaler(enabled=USE_AMP)

RESUME_FILE = os.path.join(OUTPUT_DIR, 'last_checkpoint.pt')
history     = []
start_epoch = 1
best_loss   = float('inf')

if os.path.exists(RESUME_FILE):
    print('Checkpoint found, continue..')
    ckpt = torch.load(RESUME_FILE, map_location=DEVICE)
    last_model_dir = os.path.join(OUTPUT_DIR, 'last_model')
    if os.path.exists(last_model_dir):
        model = AutoModelForObjectDetection.from_pretrained(
            last_model_dir, id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True
        ).to(DEVICE)
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_loss   = ckpt['best_loss']
    history     = ckpt['history']
    print(f'Continue from Epoch {ckpt["epoch"]}\". Best loss: {best_loss:.4f}')
else:
    print('Training starts from beginning.')


def save_checkpoint(epoch):
    model.save_pretrained(os.path.join(OUTPUT_DIR, 'last_model'))
    processor.save_pretrained(os.path.join(OUTPUT_DIR, 'last_model'))
    torch.save({
        'epoch': epoch, 'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'best_loss': best_loss, 'history': history,
    }, RESUME_FILE)

print(f'total steps: {total_steps:,} | Warmup: {warmup_steps:,}')

## Training

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for step, (pixel_values, pixel_mask, labels) in enumerate(loader):
        pixel_values = pixel_values.to(device, non_blocking=True)
        pixel_mask   = pixel_mask.to(device,   non_blocking=True)
        labels       = [{k: v.to(device, non_blocking=True)
                         for k, v in lbl.items()} for lbl in labels]

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            outputs = model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        if (step + 1) % 100 == 0:
            elapsed = time.time() - t0
            avg     = total_loss / (step + 1)
            eta     = (elapsed / (step+1)) * (len(loader) - step - 1) / 60
            print(f'  E{epoch} | {step+1}/{len(loader)} '
                  f'| Loss: {avg:.4f} | ETA: {eta:.1f} dk')

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total = 0.0
    for pixel_values, pixel_mask, labels in loader:
        pixel_values = pixel_values.to(device, non_blocking=True)
        pixel_mask   = pixel_mask.to(device,   non_blocking=True)
        labels       = [{k: v.to(device) for k, v in lbl.items()} for lbl in labels]
        with autocast(enabled=USE_AMP):
            out = model(pixel_values=pixel_values,
                        pixel_mask=pixel_mask, labels=labels)
        total += out.loss.item()
    return total / len(loader)


for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f'\n{"═"*60}')
    print(f'  EPOCH {epoch}/{NUM_EPOCHS}')
    print(f'{"═"*60}')

    t_epoch = time.time()
    tr_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, DEVICE, epoch
    )
    vl_loss = validate(model, val_loader, DEVICE)
    epoch_min = (time.time() - t_epoch) / 60

    print(f'\n  ✔ Train: {tr_loss:.4f}  Val: {vl_loss:.4f}  ({epoch_min:.1f} dk)')
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': vl_loss})

    if vl_loss < best_loss:
        best_loss = vl_loss
        best_dir  = os.path.join(OUTPUT_DIR, 'best_model')
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        print(f' Best model! val_loss={best_loss:.4f}')

    save_checkpoint(epoch)
    if epoch % SAVE_EVERY == 0:
        ep_dir = os.path.join(OUTPUT_DIR, f'epoch_{epoch:02d}')
        model.save_pretrained(ep_dir)
        processor.save_pretrained(ep_dir)
        print(f'Checkpoint → {ep_dir}')

print('\n Training completed!')

In [8]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

best_dir   = os.path.join(OUTPUT_DIR, 'best_model')
model_eval = AutoModelForObjectDetection.from_pretrained(best_dir).to(DEVICE)
model_eval.eval()
print('Best model saved.')

CONF_THRESH  = 0.3
coco_results = []

with torch.no_grad():
    for pixel_values, pixel_mask, labels in val_loader:
        pixel_values = pixel_values.to(DEVICE)
        pixel_mask   = pixel_mask.to(DEVICE)

        with autocast(enabled=USE_AMP):
            outputs = model_eval(pixel_values=pixel_values, pixel_mask=pixel_mask)

        target_sizes = torch.stack([
            torch.tensor([lbl['orig_size'][0].item(),
                          lbl['orig_size'][1].item()]) for lbl in labels
        ])
        results = processor.post_process_object_detection(
            outputs, threshold=CONF_THRESH, target_sizes=target_sizes
        )
        for lbl, res in zip(labels, results):
            img_id = lbl['image_id'].item()
            for score, cat_id, box in zip(res['scores'], res['labels'], res['boxes']):
                x1, y1, x2, y2 = box.tolist()
                coco_results.append({
                    'image_id'   : img_id,
                    'category_id': int(cat_id),
                    'bbox'       : [x1, y1, x2-x1, y2-y1],
                    'score'      : float(score),
                })

print(f'total prediction: {len(coco_results):,}')

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

Best model saved.


NameError: name 'val_loader' is not defined

##Loss Curve

In [ ]:
epochs_v = [h['epoch']      for h in history]
train_v  = [h['train_loss'] for h in history]
val_v    = [h['val_loss']   for h in history]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs_v, train_v, 'o-', label='Train', color='steelblue')
ax.plot(epochs_v, val_v,   's-', label='Val',   color='coral')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title(f'SAHI ({SLICE_H}×{SLICE_W}) + Deformable DETR — Training Curve')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=150)
plt.show()

##Category-wise Average Precision


In [ ]:
if len(coco_results) == 0:
    print('No prediction. Lower CONF_THRESH value (örn: 0.1).')
else:
    coco_gt = COCO(VAL_SLICED_JSON)

    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(coco_results, f)
        tmp = f.name

    coco_dt   = coco_gt.loadRes(tmp)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()

    print('\n' + '═'*60)
    print('  COCO General metrics (tile-level)')
    print('═'*60)
    coco_eval.summarize()

    precision = coco_eval.eval['precision']
    cat_ids   = coco_gt.getCatIds()
    cat_names = [coco_gt.loadCats([c])[0]['name'] for c in cat_ids]

    def mean_valid(arr):
        v = arr[arr > -1]
        return float(v.mean()) if len(v) else 0.0

    rows = []
    for k, (cid, cname) in enumerate(zip(cat_ids, cat_names)):
        rows.append({
            'Class'     : cname,
            'AP@.50:.95': mean_valid(precision[:, :, k, 0, 2]),
            'AP@.50'    : mean_valid(precision[0, :, k, 0, 2]),
            'AP@.75'    : mean_valid(precision[5, :, k, 0, 2]),
        })

    print('\n' + '═'*60)
    print('  Category-wise Average Precision  [SAHI tile-level]')
    print('═'*60)
    print(f'{"Class":<12} {"AP@.50:.95":>12} {"AP@.50":>8} {"AP@.75":>8}')
    print('─'*60)
    for r in rows:
        print(f'{r["Class"]:<12} {r["AP@.50:.95"]:>12.4f} '
              f'{r["AP@.50"]:>8.4f} {r["AP@.75"]:>8.4f}')
    print('─'*60)
    mean_ap = sum(r['AP@.50:.95'] for r in rows) / len(rows)
    print(f'{"mAP":<12} {mean_ap:>12.4f}')
    print('═'*60)

